In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
import os

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import mlflow.sklearn
import dagshub

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE


user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("DAGSHUB_USERNAME")

repo_name = "ml_ass2"

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{repo_name}.mlflow")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN


print("Connected to DagsHub + MLflow successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 77.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

class DropHighNaN(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold

    def fit(self, X, y=None):
        df_ = pd.DataFrame(X)
        miss = df_.isnull().mean()
        self.drop_cols_ = miss[miss > self.threshold].index.tolist()
        return self

    def transform(self, X):
        df_ = pd.DataFrame(X)
        return df_.drop(columns=[c for c in self.drop_cols_ if c in df_.columns])

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        df_ = pd.DataFrame(X)
        corr_matrix = df_.corr().abs()
        upper_tri   = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.drop_cols_ = [
            col for col in upper_tri.columns
            if any(upper_tri[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        df_ = pd.DataFrame(X)
        return df_.drop(
            columns=[c for c in self.drop_cols_ if c in df_.columns]
        )

class XGBPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df_ = pd.DataFrame(X)
        self.cat_cols_    = df_.select_dtypes(include='object').columns.tolist()
        self.card1_mean_  = df_.groupby('card1')['TransactionAmt'].mean().to_dict()
        self.card1_count_ = df_.groupby('card1')['TransactionAmt'].count().to_dict()
        self.card1_std_   = df_.groupby('card1')['TransactionAmt'].std().to_dict()
        self.addr1_mean_  = df_.groupby('addr1')['TransactionAmt'].mean().to_dict()
        self.addr1_count_ = df_.groupby('addr1')['TransactionAmt'].count().to_dict()

        df_cat = df_[self.cat_cols_].fillna('missing')
        self.enc_ = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            dtype='float32'
        )
        self.enc_.fit(df_cat)
        return self

    def transform(self, X):
        df_ = pd.DataFrame(X).copy()

        for col in self.cat_cols_:
            if col not in df_.columns:
                df_[col] = 'missing'
            else:
                df_[col] = df_[col].fillna('missing')

        df_[self.cat_cols_] = self.enc_.transform(df_[self.cat_cols_])

        new_cols = {
            'amt_log':        np.log1p(df_['TransactionAmt']),
            'tx_hour':        (df_['TransactionDT'] // 3600) % 24,
            'tx_dayofweek':   (df_['TransactionDT'] // (3600 * 24)) % 7,
            'tx_is_night':    ((df_['TransactionDT'] // 3600) % 24).apply(
                               lambda h: 1 if h < 6 or h >= 22 else 0).astype('int8'),
            'card1_mean_amt': df_['card1'].map(self.card1_mean_).fillna(0),
            'card1_tx_count': df_['card1'].map(self.card1_count_).fillna(0),
            'card1_std_amt':  df_['card1'].map(self.card1_std_).fillna(0),
            'addr1_mean_amt': df_['addr1'].map(self.addr1_mean_).fillna(0),
            'addr1_tx_count': df_['addr1'].map(self.addr1_count_).fillna(0),
        }
        df_ = pd.concat([df_, pd.DataFrame(new_cols, index=df_.index)], axis=1)

        return df_.copy()


class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df_ = pd.DataFrame(X)
        keep = [c for c in self.cols if c in df_.columns]
        return df_[keep].values

In [4]:
test_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")

X_test_raw = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
).set_index("TransactionID")

train_cols = pd.read_csv(
    "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv", nrows=1
).merge(
    pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv", nrows=1),
    on="TransactionID", how="left"
).drop(columns=["TransactionID", "isFraud"]).columns.tolist()

missing_cols = [c for c in train_cols if c not in X_test_raw.columns]
for col in missing_cols:
    X_test_raw[col] = np.nan

X_test_raw = X_test_raw[train_cols]

print(f"Test shape: {X_test_raw.shape}")
print(f"Missing cols added: {len(missing_cols)}")

Test shape: (506691, 432)
Missing cols added: 38


In [5]:
model_uri = "models:/XGB_FinalPipeline@champion"
pipeline  = mlflow.sklearn.load_model(model_uri)

print("Pipeline loaded!")
print(f"Steps: {[s[0] for s in pipeline.steps]}")

Pipeline loaded!
Steps: ['drop_nan', 'preprocessor', 'corr_filter', 'selector', 'model']


In [6]:
y_pred_proba = pipeline.predict_proba(
    X_test_raw.reset_index(drop=True)
)[:, 1]

print(f"Predictions shape : {y_pred_proba.shape}")
print(f"min={y_pred_proba.min():.4f} | max={y_pred_proba.max():.4f} | mean={y_pred_proba.mean():.4f}")

Predictions shape : (506691,)
min=0.0000 | max=0.9999 | mean=0.0756


In [7]:
submission = pd.DataFrame({
    "TransactionID": X_test_raw.index,
    "isFraud":       y_pred_proba
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")
print(submission.head())

submission.csv saved!
   TransactionID   isFraud
0        3663549  0.001188
1        3663550  0.002606
2        3663551  0.011122
3        3663552  0.002246
4        3663553  0.001652
